# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library. We will use [Croissant](https://mlcommons.org/croissant/) metadata to access, inspect, and process the data via its unique `@id` references for each key object.

### Dataset Source
The dataset Croissant schema is available here:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Ensure the mlcroissant library is installed (run once per environment)
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and initialize a `mlcroissant.Dataset` object from the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access and display main metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\n\nDescription: {meta.description}\n\nCitation: {meta.citeAs}\n\nLicense: {meta.license}\n")

## 2. Data Overview
Review the available record sets, their `@id`s, and their field `@id`s.

We will inspect the record sets and fields as defined in the Croissant metadata. Note: all data entities are referenced by their `@id`.

In [ ]:
# List available record sets
record_sets = list(dataset.list_record_sets())
print('Available Record Sets and their @id:')
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs['name'] if 'name' in rs else '(no name)'}")

# Example: show fields for each record set
record_set_fields = dict()
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    fields = rs.get('fields', [])
    if fields:
        print("  Field @id's:")
        for field in fields:
            if isinstance(field, dict) and '@id' in field:
                print(f"    - {field['@id']}")
        record_set_fields[rs['@id']] = [field['@id'] if isinstance(field, dict) and '@id' in field else field for field in fields]
    else:
        print("  (No fields found)")

## 3. Data Extraction
Extract records from each record set. We reference record sets and field columns by their `@id`.

We'll load all record sets defined in the Croissant schema into pandas DataFrames for further analysis.

In [ ]:
# Extract and load all record sets present in the schema
dfs = dict()

for rs in record_sets:
    rs_id = rs['@id']
    try:
        recs = list(dataset.records(record_set=rs_id))
        if recs:
            df = pd.DataFrame(recs)
            dfs[rs_id] = df
            print(f"\nLoaded records for Record Set: {rs_id}")
            print(f"Columns (@id): {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"\nNo records found in Record Set: {rs_id}")
    except Exception as e:
        print(f"\nCould not load records for Record Set: {rs_id}\nError: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering and transformations. For this demonstration, we'll select a numeric field from one of the populated record sets and perform filtering, normalization, and grouping.

Update the variable values as appropriate to match the actual `@id` found in your previous overview.

In [ ]:
# Example: If a record set contains regression results, select a numeric field (e.g., log_likelihood or a coefficient field) for EDA
# First, inspect what dataframes were created:
print('Available DataFrames by Record Set @id:')
for k in dfs:
    print(f"  - {k}: columns={dfs[k].columns.tolist()}")

# Select the first record set as an example (customize as needed)
record_set_id = None
numeric_field_id = None
group_field_id = None

for rs_id, df in dfs.items():
    # Try to find a numeric column (float or int) to use for EDA
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            record_set_id = rs_id
            numeric_field_id = col
            break
    if record_set_id:
        break

if record_set_id and numeric_field_id:
    print(f"\nPerforming EDA on Record Set: {record_set_id}, Numeric Field: {numeric_field_id}")
    # Optionally, find a non-numeric/grouping field for demonstration
    for col in dfs[record_set_id].columns:
        if not pd.api.types.is_numeric_dtype(dfs[record_set_id][col]):
            group_field_id = col
            break
    threshold = dfs[record_set_id][numeric_field_id].mean() if dfs[record_set_id][numeric_field_id].mean() is not None else 0
    filtered_df = dfs[record_set_id][dfs[record_set_id][numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} above mean (threshold={threshold}):\n", filtered_df.head())
    # Normalize the numeric field
    normed_field = f"{numeric_field_id}_normalized"
    filtered_df[normed_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:\n", filtered_df[[numeric_field_id, normed_field]].head())
    # Group by another field (if present)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:\n", grouped.head())
else:
    print("No suitable numeric field found for EDA. Please inspect the dataset contents.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, how it varies by a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Make a histogram and (optionally) box plot
if record_set_id and numeric_field_id:
    df = dfs[record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    # If grouping field is available and not too high cardinality, draw boxplot
    if group_field_id and df[group_field_id].nunique() < 20:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion

- We demonstrated loading and exploring the FAIR^2 dataset with `mlcroissant`.
- Record sets and fields were accessed and referenced by their `@id`, following Croissant standards.
- We performed basic EDA and visualizations for available numeric fields.
- For further exploration, repeat this process for other record sets/fields or consult the Croissant schema documentation for field descriptions.

_This workflow ensures gender, geography, and socio-demographic characteristics affecting rangeland knowledge adoption can be robustly analyzed from well-described metadata and records._